In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import dataset
df = pd.read_csv('../data/marketing_campaign.csv', sep='\t')

# 1. Remove unnecessary columns
df.drop(columns=['ID', 'Z_CostContact', 'Z_Revenue', 'Response'], inplace=True)


# 2. Feature engineering with dates
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
df['Customer_Since'] = (pd.to_datetime('today') - df['Dt_Customer']).dt.days
df['Age'] = (pd.to_datetime('today') - pd.to_datetime(df['Year_Birth'], format='%Y')).dt.days // 365
df.drop(columns=['Dt_Customer', 'Year_Birth'], inplace=True)

missing_value_columns = (df.isnull().sum().sort_values(ascending=False))
for column, null_count in missing_value_columns.items():
    if null_count > 0:
        if(column in ['Income', 'Kidhome', 'Teenhome']):
            df[column] = df[column].fillna(df[column].median())
        else:
            df[column] = df[column].fillna(df[column].mode()[0])

# 3. Identify numeric and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

# 4. Create pipelines for numeric and categorical features
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 5. Combine pipelines into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)




Numerical columns: Index(['Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Customer_Since', 'Age'],
      dtype='str') 23
Categorical columns: Index(['Education', 'Marital_Status'], dtype='str') 2


/var/folders/_7/y309r1qn27z2xg69gq1_hsw80000gn/T/ipykernel_77072/1520218769.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns
